<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-39.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 39 - Ganglia + Evaluación BDA-RA4

**Módulo:** Big Data Aplicado (BDA - 5075)

**Bloque:** RA4 - Monitorización

---

## Objetivos del notebook

1. Simular un dashboard tipo Ganglia con datos de clúster
2. Implementar consolidación de series temporales (como RRDtool)
3. Detectar umbrales y generar alertas automáticas
4. Ejercicios de práctica para la evaluación RA4

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

print("=== Librerías cargadas ===")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print("matplotlib, datetime, random")

---

## 1. Simulación de datos de gmond

Ganglia recolecta métricas con gmond en cada nodo.
Vamos a simular las métricas que gmond reportaría durante 48 horas
en un clúster Hadoop de 6 nodos.

In [ ]:
NODOS = {
    'namenode':     {'rack': 'rack-01', 'role': 'master', 'cpu_cores': 16, 'ram_gb': 64},
    'resourcemgr':  {'rack': 'rack-01', 'role': 'master', 'cpu_cores': 16, 'ram_gb': 64},
    'datanode-01':  {'rack': 'rack-01', 'role': 'worker', 'cpu_cores': 8,  'ram_gb': 32},
    'datanode-02':  {'rack': 'rack-01', 'role': 'worker', 'cpu_cores': 8,  'ram_gb': 32},
    'datanode-03':  {'rack': 'rack-02', 'role': 'worker', 'cpu_cores': 8,  'ram_gb': 32},
    'datanode-04':  {'rack': 'rack-02', 'role': 'worker', 'cpu_cores': 8,  'ram_gb': 32},
}

INTERVALO_SEG = 15
DURACION_HORAS = 48
inicio = datetime(2026, 3, 10, 0, 0, 0)
n_muestras = (DURACION_HORAS * 3600) // INTERVALO_SEG

timestamps = [inicio + timedelta(seconds=i * INTERVALO_SEG) for i in range(n_muestras)]

registros = []
for nodo, info in NODOS.items():
    cpu_base = 25 if info['role'] == 'master' else 35
    mem_base = 50 if info['role'] == 'master' else 45

    for i, ts in enumerate(timestamps):
        hora = ts.hour
        factor_hora = 1.4 if 9 <= hora <= 17 else (0.6 if 0 <= hora <= 6 else 1.0)

        # Evento: job grande entre 10:00 y 14:00 del día 2
        factor_job = 1.0
        if ts.day == 11 and 10 <= hora <= 14 and info['role'] == 'worker':
            factor_job = 2.0

        # Evento: datanode-03 empieza a fallar a las 18h del día 2
        factor_fallo = 1.0
        if nodo == 'datanode-03' and ts.day == 11 and hora >= 18:
            progreso = (hora - 18) / 6.0
            factor_fallo = 1.0 + progreso * 2.5

        cpu_user = np.clip(cpu_base * factor_hora * factor_job * factor_fallo
                           + np.random.normal(0, 4), 0, 100)
        cpu_system = np.clip(5 + np.random.normal(0, 2) * factor_fallo, 0, 30)
        cpu_idle = max(0, 100 - cpu_user - cpu_system)
        load_one = max(0, (cpu_user / 100) * info['cpu_cores'] * 1.2 + np.random.normal(0, 0.5))
        mem_used_pct = np.clip(mem_base * factor_hora * factor_job * factor_fallo
                               + np.random.normal(0, 3), 0, 100)
        disk_total_gb = 1000 if info['role'] == 'worker' else 500
        disk_used_pct = np.clip(55 + (i / n_muestras) * 8 + np.random.normal(0, 0.5), 0, 100)
        bytes_in = max(0, 50_000_000 * factor_hora * factor_job + np.random.normal(0, 5_000_000))
        bytes_out = max(0, 30_000_000 * factor_hora * factor_job + np.random.normal(0, 3_000_000))

        registros.append({
            'timestamp': ts, 'nodo': nodo, 'rack': info['rack'],
            'cpu_user': round(cpu_user, 1), 'cpu_system': round(cpu_system, 1),
            'cpu_idle': round(cpu_idle, 1), 'load_one': round(load_one, 2),
            'mem_used_pct': round(mem_used_pct, 1),
            'disk_used_pct': round(disk_used_pct, 1),
            'bytes_in': int(bytes_in), 'bytes_out': int(bytes_out)
        })

df = pd.DataFrame(registros)

print(f"Datos simulados de gmond: {len(df):,} registros")
print(f"Periodo: {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"Nodos: {df['nodo'].nunique()} | Intervalo: {INTERVALO_SEG}s")
print(f"\nMétricas por nodo: {len(df) // len(NODOS):,}")
df.head()

---

## 2. Dashboard estilo Ganglia

Creamos una vista similar a la que ofrece gweb (Ganglia Web Frontend),
mostrando métricas agregadas del clúster y por nodo.

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle('Ganglia - Cluster View: hadoop-cluster (48h)', fontsize=16, fontweight='bold')
gs = gridspec.GridSpec(3, 2, hspace=0.35, wspace=0.25)

colores_nodo = {
    'namenode': '#e74c3c', 'resourcemgr': '#c0392b',
    'datanode-01': '#3498db', 'datanode-02': '#2ecc71',
    'datanode-03': '#f39c12', 'datanode-04': '#9b59b6'
}

# CPU User por nodo
ax1 = fig.add_subplot(gs[0, 0])
for nodo in NODOS:
    datos = df[df['nodo'] == nodo]
    resampled = datos.set_index('timestamp')['cpu_user'].resample('15min').mean()
    ax1.plot(resampled.index, resampled.values, label=nodo,
             color=colores_nodo[nodo], alpha=0.8, linewidth=0.8)
ax1.axhline(y=80, color='red', linestyle='--', alpha=0.4, linewidth=1)
ax1.set_title('cpu_user (%)', fontweight='bold')
ax1.legend(fontsize=6, ncol=2)
ax1.grid(True, alpha=0.2)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
ax1.tick_params(axis='x', rotation=30, labelsize=7)

# Load Average (1 min)
ax2 = fig.add_subplot(gs[0, 1])
for nodo in NODOS:
    datos = df[df['nodo'] == nodo]
    resampled = datos.set_index('timestamp')['load_one'].resample('15min').mean()
    ax2.plot(resampled.index, resampled.values, label=nodo,
             color=colores_nodo[nodo], alpha=0.8, linewidth=0.8)
ax2.set_title('load_one (1-min avg)', fontweight='bold')
ax2.legend(fontsize=6, ncol=2)
ax2.grid(True, alpha=0.2)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
ax2.tick_params(axis='x', rotation=30, labelsize=7)

# Memoria agregada del clúster
ax3 = fig.add_subplot(gs[1, 0])
mem_cluster = df.groupby('timestamp')['mem_used_pct'].mean()
mem_resampled = mem_cluster.resample('15min').mean()
ax3.fill_between(mem_resampled.index, 0, mem_resampled.values, alpha=0.3, color='#3498db')
ax3.plot(mem_resampled.index, mem_resampled.values, color='#2980b9', linewidth=1)
ax3.axhline(y=85, color='red', linestyle='--', alpha=0.4)
ax3.set_title('mem_used_pct - Cluster Average (%)', fontweight='bold')
ax3.set_ylim(0, 100)
ax3.grid(True, alpha=0.2)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
ax3.tick_params(axis='x', rotation=30, labelsize=7)

# Disco por nodo (solo workers)
ax4 = fig.add_subplot(gs[1, 1])
for nodo in ['datanode-01', 'datanode-02', 'datanode-03', 'datanode-04']:
    datos = df[df['nodo'] == nodo]
    resampled = datos.set_index('timestamp')['disk_used_pct'].resample('15min').mean()
    ax4.plot(resampled.index, resampled.values, label=nodo,
             color=colores_nodo[nodo], alpha=0.8, linewidth=1)
ax4.axhline(y=75, color='orange', linestyle='--', alpha=0.4, label='Warning (75%)')
ax4.axhline(y=85, color='red', linestyle='--', alpha=0.4, label='Critical (85%)')
ax4.set_title('disk_used_pct - DataNodes (%)', fontweight='bold')
ax4.legend(fontsize=6)
ax4.grid(True, alpha=0.2)
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
ax4.tick_params(axis='x', rotation=30, labelsize=7)

# Red: bytes_in agregado
ax5 = fig.add_subplot(gs[2, 0])
net_in = df.groupby('timestamp')['bytes_in'].sum() / 1e6
net_resampled = net_in.resample('15min').mean()
ax5.fill_between(net_resampled.index, 0, net_resampled.values, alpha=0.3, color='#27ae60')
ax5.plot(net_resampled.index, net_resampled.values, color='#27ae60', linewidth=1)
ax5.set_title('bytes_in - Cluster Total (MB/s)', fontweight='bold')
ax5.grid(True, alpha=0.2)
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
ax5.tick_params(axis='x', rotation=30, labelsize=7)

# Resumen por rack
ax6 = fig.add_subplot(gs[2, 1])
rack_cpu = df.groupby(['rack', pd.Grouper(key='timestamp', freq='1h')])['cpu_user'].mean().reset_index()
for rack in ['rack-01', 'rack-02']:
    datos_rack = rack_cpu[rack_cpu['rack'] == rack]
    ax6.plot(datos_rack['timestamp'], datos_rack['cpu_user'],
             label=rack, linewidth=1.5, alpha=0.8)
ax6.set_title('cpu_user por Rack - Media horaria (%)', fontweight='bold')
ax6.legend()
ax6.grid(True, alpha=0.2)
ax6.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
ax6.tick_params(axis='x', rotation=30, labelsize=7)

plt.show()

print("📊 Dashboard tipo Ganglia generado")
print("   Observa el pico de carga (10h-14h día 11) y el fallo de datanode-03 (18h+ día 11)")

---

## 3. Simulación de consolidación RRD

RRDtool almacena datos con diferentes resoluciones:
alta resolución para datos recientes, baja para datos antiguos.
Simulamos esta política de retención.

In [ ]:
datos_namenode = df[df['nodo'] == 'namenode'].set_index('timestamp')['cpu_user']

resolucion_15s = datos_namenode
resolucion_5min = datos_namenode.resample('5min').mean()
resolucion_30min = datos_namenode.resample('30min').mean()
resolucion_2h = datos_namenode.resample('2h').mean()

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
fig.suptitle('Consolidación RRDtool - Namenode CPU', fontsize=14, fontweight='bold')

resoluciones = [
    (resolucion_15s, '15 seg (datos crudos)', '#e74c3c', axes[0]),
    (resolucion_5min, '5 min (consolidado)', '#e67e22', axes[1]),
    (resolucion_30min, '30 min (consolidado)', '#2ecc71', axes[2]),
    (resolucion_2h, '2 horas (consolidado)', '#3498db', axes[3]),
]

for datos, titulo, color, ax in resoluciones:
    ax.plot(datos.index, datos.values, color=color, linewidth=0.8, alpha=0.8)
    ax.fill_between(datos.index, 0, datos.values, color=color, alpha=0.15)
    ax.set_ylabel('CPU %')
    ax.set_title(f'Resolución: {titulo} ({len(datos):,} puntos)', fontsize=10)
    ax.grid(True, alpha=0.2)
    ax.set_ylim(0, 60)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
axes[-1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print("📊 Observa cómo la consolidación suaviza las métricas:")
print(f"   15s:   {len(resolucion_15s):>6,} puntos → detalle máximo, mucho espacio")
print(f"   5min:  {len(resolucion_5min):>6,} puntos → buen balance")
print(f"   30min: {len(resolucion_30min):>6,} puntos → tendencias claras")
print(f"   2h:    {len(resolucion_2h):>6,} puntos → solo tendencias macro")
print(f"\n💡 RRDtool hace esto automáticamente según la política de retención.")

---

## 4. Sistema de alertas con umbrales

Implementamos detección de anomalías combinando umbrales estáticos
(como los que configuraríamos con Nagios) y detección de cambios bruscos.

In [ ]:
ALERTAS_CONFIG = {
    'cpu_user':     {'warning': 70, 'critical': 90, 'nombre': 'CPU User'},
    'mem_used_pct': {'warning': 80, 'critical': 95, 'nombre': 'Memoria'},
    'disk_used_pct':{'warning': 75, 'critical': 85, 'nombre': 'Disco'},
    'load_one':     {'warning': 8,  'critical': 14, 'nombre': 'Load Average'},
}

df_5min = df.copy()
df_5min['timestamp_5min'] = df_5min['timestamp'].dt.floor('5min')
df_5min = df_5min.groupby(['timestamp_5min', 'nodo']).agg({
    'cpu_user': 'mean', 'cpu_system': 'mean', 'mem_used_pct': 'mean',
    'disk_used_pct': 'mean', 'load_one': 'mean', 'rack': 'first'
}).reset_index()

alertas = []
for _, row in df_5min.iterrows():
    for metrica, config in ALERTAS_CONFIG.items():
        valor = row[metrica]
        if valor >= config['critical']:
            alertas.append({
                'timestamp': row['timestamp_5min'], 'nodo': row['nodo'],
                'rack': row['rack'], 'metrica': config['nombre'],
                'valor': round(valor, 1), 'nivel': 'CRITICAL'
            })
        elif valor >= config['warning']:
            alertas.append({
                'timestamp': row['timestamp_5min'], 'nodo': row['nodo'],
                'rack': row['rack'], 'metrica': config['nombre'],
                'valor': round(valor, 1), 'nivel': 'WARNING'
            })

df_alertas = pd.DataFrame(alertas)

print("=" * 65)
print("SISTEMA DE ALERTAS - RESUMEN")
print("=" * 65)
print(f"\nTotal de alertas generadas: {len(df_alertas)}")

print("\n--- Por nivel ---")
print(df_alertas['nivel'].value_counts().to_string())

print("\n--- Por nodo y nivel ---")
tabla = df_alertas.groupby(['nodo', 'nivel']).size().unstack(fill_value=0)
print(tabla.to_string())

print("\n--- Por métrica ---")
print(df_alertas.groupby(['metrica', 'nivel']).size().unstack(fill_value=0).to_string())

print("\n--- Alertas CRITICAL más recientes ---")
criticas = df_alertas[df_alertas['nivel'] == 'CRITICAL'].sort_values('timestamp', ascending=False)
for _, a in criticas.head(10).iterrows():
    print(f"  🔴 [{a['timestamp']}] {a['nodo']}: {a['metrica']} = {a['valor']}")

print("\n⚠️  ¿datanode-03 concentra la mayoría de alertas CRITICAL? ¿Por qué?")

---

## 5. Ejercicio de práctica para la evaluación

### Escenario

Te proporcionan un CSV con métricas de un clúster.
Debes analizarlo, encontrar problemas y proponer soluciones.
Este ejercicio es similar a la **Parte C** de la evaluación.

In [ ]:
csv_path = '/tmp/metricas_cluster_examen.csv'
df_5min.rename(columns={'timestamp_5min': 'timestamp'}).to_csv(csv_path, index=False)

print("=== EJERCICIO PRÁCTICO (tipo examen) ===")
print(f"\nFichero: {csv_path}")
print("\nTareas:")
print("  1. Carga el CSV y muestra las primeras filas")
print("  2. ¿Cuántos nodos hay? ¿Cuál es el periodo temporal?")
print("  3. Calcula la media de CPU para cada nodo")
print("  4. Encuentra el nodo con mayor uso medio de memoria")
print("  5. Detecta todos los momentos donde CPU > 80%")
print("  6. Genera un gráfico de CPU por nodo")
print("\n" + "=" * 50)

# --- SOLUCIÓN ---

print("\n--- 1. Carga del CSV ---")
df_exam = pd.read_csv(csv_path, parse_dates=['timestamp'])
print(f"Shape: {df_exam.shape}")
print(df_exam.head())

print("\n--- 2. Exploración ---")
print(f"Nodos: {df_exam['nodo'].unique()}")
print(f"Periodo: {df_exam['timestamp'].min()} → {df_exam['timestamp'].max()}")

print("\n--- 3. Media de CPU por nodo ---")
cpu_media = df_exam.groupby('nodo')['cpu_user'].mean().round(1).sort_values(ascending=False)
print(cpu_media.to_string())

print("\n--- 4. Nodo con mayor uso de memoria ---")
mem_media = df_exam.groupby('nodo')['mem_used_pct'].mean().round(1)
nodo_max_mem = mem_media.idxmax()
print(f"Nodo con más memoria: {nodo_max_mem} ({mem_media[nodo_max_mem]}%)")

print("\n--- 5. Alertas de CPU > 80% ---")
alertas_cpu = df_exam[df_exam['cpu_user'] > 80][['timestamp', 'nodo', 'cpu_user']]
print(f"Total de registros con CPU > 80%: {len(alertas_cpu)}")
print(alertas_cpu.groupby('nodo').size().to_string())

In [ ]:
print("--- 6. Gráfico de CPU por nodo ---\n")

fig, ax = plt.subplots(figsize=(14, 5))

for nodo in df_exam['nodo'].unique():
    datos = df_exam[df_exam['nodo'] == nodo]
    ax.plot(datos['timestamp'], datos['cpu_user'], label=nodo, alpha=0.7, linewidth=0.8)

ax.axhline(y=80, color='red', linestyle='--', alpha=0.5, label='Umbral (80%)')
ax.set_title('CPU User por nodo', fontweight='bold')
ax.set_xlabel('Timestamp')
ax.set_ylabel('CPU User (%)')
ax.legend(fontsize=7, ncol=3)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

print("✅ Ejercicio completado.")
print("   En el examen, asegúrate de incluir interpretación de los resultados.")
print("   Ejemplo: 'datanode-03 muestra un incremento anómalo a partir de las 18h,")
print("   lo que sugiere un problema de hardware o un proceso descontrolado.'")

---

## Resumen

| Concepto | Herramienta real | Simulación Python |
|----------|-----------------|-------------------|
| Recolección de métricas | gmond | `random` + `pandas` |
| Agregación y almacenamiento | gmetad + RRDtool | `df.resample()` |
| Visualización | gweb | `matplotlib` dashboard |
| Alertas | Nagios / Alertmanager | Umbrales + `pandas` |
| Consolidación temporal | RRD Round Robin | `resample()` a diferentes frecuencias |

### Conceptos clave para la evaluación

1. **Ganglia** = gmond (nodo) + gmetad (central) + gweb (web)
2. **RRDtool** = tamaño fijo, consolidación automática, alta resolución reciente
3. **Métricas Hadoop** = hadoop-metrics2.properties → Ganglia
4. **Alertas** = umbrales warning/critical + acciones correctivas
5. **Análisis de logs** = niveles (FATAL > ERROR > WARN > INFO) + patrones con regex

---

*¡Buena suerte en la evaluación!* 🍀